# Erros e aritmética de ponto flutuante

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/02-erros.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. explicar como um computador guarda números reais e por que $0{,}1 + 0{,}2 \neq 0{,}3$;
> 2. distinguir erro de **arredondamento** de erro de **truncamento** e prever o passo ótimo quando os dois competem;
> 3. separar um problema **mal condicionado** de um algoritmo **instável**;
> 4. reconhecer e corrigir o cancelamento catastrófico em código, seu ou de um agente.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

## 1. Problema de engenharia: o relógio do míssil Patriot

Em 25 de fevereiro de 1991, durante a Guerra do Golfo, uma bateria de mísseis Patriot em Dhahran, na Arábia Saudita, não interceptou um míssil Scud. O Scud atingiu um quartel e matou 28 soldados. A investigação do governo americano (relatório GAO/IMTEC-92-26) encontrou a causa num detalhe aritmético.

O relógio do sistema era um **contador inteiro**: a cada décimo de segundo, ele somava 1. Depois de uma hora, por exemplo, o contador valia 36 000. Essa contagem é exata, porque inteiros são representados sem erro no computador. Para obter o tempo em segundos, o programa multiplicava o contador por $0{,}1$, e o problema estava nesse $0{,}1$: ele **não tem representação exata em binário**. Guardado num registrador de 24 bits, ele vale cerca de $0{,}1 - 9{,}5\times 10^{-8}$. Esse pequeno erro é multiplicado pelo contador, que cresce sem parar. Um erro de cem bilionésimos parece inofensivo. Mas a bateria estava ligada havia **100 horas**:

$$
100 \text{ h} \times 3600 \tfrac{\text{s}}{\text{h}} \times 10 \tfrac{\text{décimos}}{\text{s}} \times 9{,}5\times 10^{-8} \text{ s} \approx 0{,}34 \text{ s}.
$$

Um Scud viaja a cerca de 1 700 m/s. Em 0,34 s, ele percorre **mais de 500 metros**, e o radar procurou o alvo no lugar errado.

Este capítulo explica de onde vêm erros como esse, como eles se propagam e como evitá-los.

In [ ]:
# O erro de representar 0,1 com 23 bits depois da vírgula binária (como no Patriot)
decimo_24bits = int(0.1 * 2**23) / 2**23
erro_por_decimo = 0.1 - decimo_24bits
decimos = 100 * 3600 * 10
print(f"0,1 guardado em 24 bits: {decimo_24bits:.12f}")
print(f"erro por décimo de segundo: {erro_por_decimo:.2e} s")
print(f"erro acumulado em 100 h:    {erro_por_decimo * decimos:.4f} s")
print(f"deslocamento do alvo:       {erro_por_decimo * decimos * 1676:.0f} m  (Scud a 1676 m/s)")

## 2. A ideia em uma imagem

O computador não guarda todos os números reais, só um conjunto finito de **números de máquina**. Entre dois números de máquina vizinhos há um buraco, e todo resultado que cai no buraco é arredondado para um dos lados. A figura mostra todos os números positivos de um sistema de brinquedo com 3 bits de mantissa.

In [ ]:
def numeros_de_maquina(bits_mantissa, e_min, e_max):
    # Todos os números positivos normalizados (1,b1 b2 ... bp)₂ × 2^e
    valores = []
    for e in range(e_min, e_max + 1):
        for k in range(2**bits_mantissa):
            valores.append((1 + k / 2**bits_mantissa) * 2.0**e)
    return np.array(valores)

nm = numeros_de_maquina(3, -4, 2)
fig = go.Figure()
fig.add_trace(go.Scatter(x=nm, y=np.zeros_like(nm), mode="markers",
                         marker=dict(symbol="line-ns-open", size=22, line=dict(width=2), color=CORES["aproximacao"]),
                         hovertemplate="%{x}<extra></extra>", name="números de máquina"))
for e in range(0, 3):
    fig.add_annotation(x=1.5 * 2.0**e, y=0.6, text=f"espaçamento {numero(2.0**e / 8)}", showarrow=False,
                       font=dict(size=11, color=CORES["referencia"]))
fig.add_trace(go.Scatter(x=[0.1], y=[0], mode="markers", marker=dict(color=CORES["erro"], size=10),
                         name="0,1 (não é número de máquina)"))
fig.add_annotation(x=0.1, y=0, ax=40, ay=-45, text="0,1", showarrow=True, arrowhead=2,
                   arrowcolor=CORES["erro"], font=dict(color=CORES["erro"]))
fig.update_xaxes(title="valor", range=[0, 8.2])
fig.update_yaxes(visible=False, range=[-1, 1.2])
fig.update_layout(height=260, title="Os números de máquina ficam mais espaçados à medida que crescem",
                  legend=dict(orientation="h", y=-0.5))
mostrar(fig)

**O que notar:** o espaçamento **dobra** a cada potência de 2. O **erro relativo** de arredondamento, porém, fica sempre abaixo de um mesmo valor, a precisão da máquina. É por isso que se fala em "algarismos significativos": o computador guarda um número fixo de dígitos, onde quer que esteja a vírgula.

## 3. Formulação matemática

### 3.1 Erro absoluto e relativo

Se $x$ é o valor verdadeiro e $\tilde x$ a aproximação,

$$
E = x - \tilde x, \qquad \varepsilon = \frac{x - \tilde x}{x}.
$$

O erro relativo é o que importa na maioria das vezes. Errar 1 cm em 1 m é ruim, mas errar 1 cm em 1 km é excelente. Um resultado com erro relativo $|\varepsilon| \le 0{,}5 \times 10^{-n}$ tem cerca de $n$ **algarismos significativos** corretos.

Nos métodos iterativos não se conhece $x$, e o erro é estimado pela variação entre iterações, $\varepsilon_a = (\tilde x_{\text{novo}} - \tilde x_{\text{velho}})/\tilde x_{\text{novo}}$. É esse o critério de parada dos métodos do capítulo 3.

### 3.2 Ponto flutuante: o padrão IEEE 754

Um número de precisão dupla (`float64`, o padrão do Python e do NumPy) ocupa 64 bits:

$$
x = (-1)^{s} \times (1{,}b_1 b_2 \dots b_{52})_2 \times 2^{\,e - 1023},
$$

com 1 bit de sinal $s$, 11 bits de expoente $e$ e 52 bits de **mantissa**. O "1," inicial não é guardado: é o bit implícito.

A grandeza central é a **precisão da máquina**, a distância entre 1 e o próximo número de máquina:

$$
\varepsilon_{\text{mach}} = 2^{-52} \approx 2{,}22 \times 10^{-16}.
$$

Todo resultado de uma operação aritmética é o resultado exato arredondado:

$$
\mathrm{fl}(a \circ b) = (a \circ b)(1 + \delta), \qquad |\delta| \le \tfrac12\varepsilon_{\text{mach}} .
$$ (eq-modelo-fl)

Cada operação introduz um erro relativo de no máximo $1{,}1\times 10^{-16}$, cerca de 16 algarismos significativos. O problema não é uma operação isolada: é o que acontece quando os erros se **acumulam** (como no Patriot) ou são **amplificados**.

### 3.3 Cancelamento catastrófico

Subtrair dois números quase iguais **amplifica** o erro relativo. Se $a$ e $b$ têm erro relativo $\delta$, então

$$
\frac{|\Delta(a - b)|}{|a - b|} \approx \delta\,\frac{|a| + |b|}{|a - b|}.
$$

Se $a$ e $b$ concordam nos primeiros 10 dígitos, $|a|/|a-b| \sim 10^{10}$, e dos 16 algarismos sobram 6. O exemplo clássico é a fórmula de Bhaskara quando $b^2 \gg 4ac$:

$$
x_1 = \frac{-b + \sqrt{b^2 - 4ac}}{2a}.
$$

Para $b > 0$, $\sqrt{b^2 - 4ac} \approx b$ e a soma no numerador cancela. A saída é reescrever a expressão. Multiplicando em cima e embaixo pelo conjugado:

$$
x_1 = \frac{-2c}{b + \sqrt{b^2 - 4ac}},
$$

que só **soma** números de mesmo sinal. A matemática é a mesma; o comportamento numérico, não.

### 3.4 Erro de truncamento e a série de Taylor

O **erro de truncamento** vem de trocar um processo infinito por um finito. A ferramenta para medi-lo é a série de Taylor com resto:

$$
f(x + h) = f(x) + f'(x)\,h + \frac{f''(x)}{2!}h^2 + \dots + \frac{f^{(n)}(x)}{n!}h^n + R_n, \qquad R_n = \frac{f^{(n+1)}(\xi)}{(n+1)!}h^{n+1}.
$$

Por exemplo, isolando $f'(x)$ com $n = 1$:

$$
f'(x) = \frac{f(x+h) - f(x)}{h} - \frac{f''(\xi)}{2}\, h .
$$

A **diferença progressiva** tem erro de truncamento $O(h)$: diminuir $h$ diminui o erro.

### 3.5 Quando os dois erros competem

Diminuir $h$ reduz o truncamento, mas a subtração $f(x+h) - f(x)$ sofre cancelamento quando $h$ é pequeno. Se cada valor de $f$ tem erro de arredondamento da ordem de $\varepsilon_{\text{mach}}|f|$, o erro total é

$$
E(h) \approx \underbrace{\frac{|f''|}{2}\,h}_{\text{truncamento}} + \underbrace{\frac{2\,\varepsilon_{\text{mach}}|f|}{h}}_{\text{arredondamento}} .
$$

O mínimo fica em $h^* \approx 2\sqrt{\varepsilon_{\text{mach}}|f/f''|} \sim 10^{-8}$, e o melhor erro possível é da ordem de $\sqrt{\varepsilon_{\text{mach}}} \approx 10^{-8}$. **Diminuir $h$ além de $h^*$ piora o resultado.** A seção 5 mostra isso num gráfico.

### 3.6 Condicionamento e estabilidade

São duas perguntas diferentes, que vale a pena separar:

- **Condicionamento** é uma propriedade do **problema**: quanto a saída muda quando a entrada muda um pouco? Para $y = f(x)$, o número de condição é

$$
\kappa(x) = \left|\frac{x\, f'(x)}{f(x)}\right|,
$$

e um erro relativo $\delta$ na entrada vira cerca de $\kappa\,\delta$ na saída. Se $\kappa$ é grande, o problema é **mal condicionado**, e nenhum algoritmo resolve isso.
- **Estabilidade** é uma propriedade do **algoritmo**: ele introduz erros muito maiores que os inevitáveis? A fórmula de Bhaskara para $x_1$ é **instável** quando $b^2 \gg 4ac$, embora o problema (achar a raiz) seja bem condicionado. A fórmula alternativa é estável.

### 3.7 Propagação de erros

Se $y = f(x_1, \dots, x_n)$ e cada $x_i$ tem incerteza $\Delta x_i$, a aproximação de primeira ordem por Taylor dá

$$
\Delta y \approx \sum_{i=1}^{n} \left|\frac{\partial f}{\partial x_i}\right| \Delta x_i .
$$

Essa fórmula é a base da análise de incertezas em medidas de engenharia. O apêndice A a estende com Monte Carlo e diferenciação automática.

## 4. À mão

**0,1 em binário.** Multiplicando a parte fracionária por 2 repetidamente e anotando a parte inteira:

| passo | valor × 2 | dígito binário |
| --- | --- | --- |
| 1 | 0,2 | 0 |
| 2 | 0,4 | 0 |
| 3 | 0,8 | 0 |
| 4 | 1,6 | 1 |
| 5 | 1,2 | 1 |
| 6 | 0,4 | 0 |
| 7 | 0,8 | 0 |
| 8 | 1,6 | 1 |

A partir do passo 6, o ciclo 0, 0, 1, 1 se repete: $0{,}1 = (0{,}0\overline{0011})_2$. Como a dízima é infinita, qualquer número finito de bits a corta, e daí vem o erro do Patriot.

**Bhaskara com 4 algarismos.** Para $x^2 + 62{,}10\,x + 1 = 0$, a raiz menor é $x_1 \approx -0{,}01610$. Com 4 algarismos significativos, $\sqrt{62{,}10^2 - 4} = \sqrt{3852} = 62{,}06$, e

$$
x_1 = \frac{-62{,}10 + 62{,}06}{2} = -0{,}02000 \quad (\text{erro de } 24\,\%), \qquad
x_1 = \frac{-2}{62{,}10 + 62{,}06} = -0{,}01611 \quad (\text{erro de } 0{,}06\,\%).
$$

In [ ]:
print(f"0,1 + 0,2 = {0.1 + 0.2!r}")
print(f"0,1 em binário (como guardado): {0.1:.55f}")
print(f"precisão da máquina: {np.finfo(float).eps:.3e}  =  2^-52 = {2.0**-52:.3e}")

## 5. Explore

### 5.1 Taylor: quantos termos são suficientes?

O slider muda o número de termos da série de Taylor de $\cos x$ em torno de $x_0 = 0$. Perto de $x_0$ poucos termos bastam; longe dele, são necessários muitos.

In [ ]:
from math import factorial

def taylor_cos(x, n):
    return sum((-1) ** k * x ** (2 * k) / factorial(2 * k) for k in range(n + 1))

def figura_taylor(n):
    x = np.linspace(-2 * np.pi, 2 * np.pi, 400)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=np.cos(x), name="cos x", line=dict(color=CORES["exata"])))
    fig.add_trace(go.Scatter(x=x, y=taylor_cos(x, n), name=f"Taylor, grau {2 * n}", line=dict(color=CORES["aproximacao"])))
    fig.update_xaxes(title="x (rad)")
    fig.update_yaxes(title="y", range=[-2, 2])
    fig.update_layout(title=f"Série de Taylor de cos x até o grau {2 * n}")
    return fig

quadros = [(str(2 * n), figura_taylor(n)) for n in range(0, 10)]
mostrar(com_controle(quadros, rotulo="grau = ", ativo=2, titulos=[q[1].layout.title.text for q in quadros]))

### 5.2 O erro total em forma de V

Erro da derivada numérica de $\sin x$ em $x = 1$, para $h$ de $10^{-1}$ até $10^{-17}$. Os pontos são o erro medido; as linhas tracejadas, as estimativas da seção 3.5.

In [ ]:
h = np.logspace(-1, -17, 65)
x0 = 1.0
erro_prog = np.abs((np.sin(x0 + h) - np.sin(x0)) / h - np.cos(x0))
erro_cent = np.abs((np.sin(x0 + h) - np.sin(x0 - h)) / (2 * h) - np.cos(x0))
eps = np.finfo(float).eps

fig = go.Figure()
fig.add_trace(go.Scatter(x=h, y=erro_prog, mode="markers", name="diferença progressiva", marker=dict(color=CORES["erro"])))
fig.add_trace(go.Scatter(x=h, y=erro_cent, mode="markers", name="diferença central", marker=dict(color=CORES["secundaria"])))
fig.add_trace(go.Scatter(x=h, y=np.sin(x0) / 2 * h, mode="lines", name="truncamento ≈ |f''| h / 2",
                         line=dict(color=CORES["referencia"], dash="dash")))
fig.add_trace(go.Scatter(x=h, y=2 * eps * np.sin(x0) / h, mode="lines", name="arredondamento ≈ 2 ε |f| / h",
                         line=dict(color=CORES["referencia"], dash="dot")))
fig.update_xaxes(type="log", title="passo h", exponentformat="power", autorange="reversed")
fig.update_yaxes(type="log", title="erro absoluto", range=[-12, 1], exponentformat="power")
fig.update_layout(title="Diminuir h ajuda até certo ponto; depois atrapalha")
mostrar(fig)
i = np.argmin(erro_prog)
print(f"melhor h (progressiva): {h[i]:.1e}, erro {erro_prog[i]:.1e}")
j = np.argmin(erro_cent)
print(f"melhor h (central):     {h[j]:.1e}, erro {erro_cent[j]:.1e}")

**O que notar:** o eixo horizontal vai de $h$ grande, à esquerda, para $h$ pequeno, à direita. À esquerda domina o truncamento: diminuindo $h$, o erro cai. À direita domina o arredondamento, e o erro **sobe** com $1/h$. O melhor $h$ fica perto de $10^{-8}$ para a diferença progressiva. A diferença central, de ordem 2, atinge um erro menor, com um $h$ maior, perto de $10^{-5}$. No extremo direito, $h < \varepsilon_{\text{mach}}/2$, então $1 + h = 1$ em ponto flutuante, a diferença progressiva dá derivada zero, e o erro é o próprio $\cos 1 \approx 0{,}54$.

## 6. Trabalhando com o agente

Um contador de frequência mede um oscilador de quartzo de referência de 10 MHz. As leituras, em mHz (milésimos de hertz), ficam perto de $10^{10}$ mHz, com flutuações de alguns mHz. Queremos a **variância** das leituras, que mede a estabilidade do oscilador.

### 6.1 O pedido e o código

> *"Escreva uma função `variancia(dados)` que calcula a variância amostral de uma lista de números, numa única passagem pelos dados, para funcionar com sequências muito longas."*

O pedido parece razoável, mas o pedido de "uma única passagem" empurra o agente para a fórmula do livro de estatística, $\operatorname{Var}(x) = \overline{x^2} - \bar x^2$:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
def variancia(dados):
    n = 0
    soma = 0.0
    soma_quadrados = 0.0
    for x in dados:
        n += 1
        soma += x
        soma_quadrados += x * x
    media = soma / n
    return (soma_quadrados - n * media**2) / (n - 1)

rng = np.random.default_rng(42)
ruido = rng.normal(0.0, 3.0, size=10_000)          # desvio padrão de 3 mHz
print("variância verdadeira ≈ 9;   código do agente com média 0:", numero(variancia(ruido), 4))

### 6.2 Verificação: um invariante

Somar uma constante a todos os dados **não muda** a variância. É um invariante matemático, e um ótimo teste. Deslocamos os dados, passo a passo, até a média real das leituras:

In [ ]:
deslocamentos = 10.0 ** np.arange(0, 11)
var_agente = [variancia(ruido + d) for d in deslocamentos]
var_numpy = [np.var(ruido + d, ddof=1) for d in deslocamentos]

fig = go.Figure()
fig.add_trace(go.Scatter(x=deslocamentos, y=var_numpy, mode="lines+markers", name="np.var (duas passagens)",
                         line=dict(color=CORES["aproximacao"])))
fig.add_trace(go.Scatter(x=deslocamentos, y=var_agente, mode="lines+markers", name="código do agente",
                         line=dict(color=CORES["erro"])))
fig.update_xaxes(type="log", title="média dos dados (mHz)", exponentformat="power")
fig.update_yaxes(title="variância calculada (mHz²)", range=[-40, 60])
fig.update_layout(title="A variância deveria ser 9 para qualquer média")
mostrar(fig)
for d, va in zip(deslocamentos[5:], var_agente[5:]):
    print(f"média {d:8.0e}:  agente = {va:12.4f}")

**O que notar:** até a média $10^6$ o erro do código do agente é pequeno, mas com $10^7$ ele já passa de 4 %. A partir de $10^8$ o código devolve uma **variância negativa**, o que é impossível. No caso do oscilador, com média $10^{10}$ mHz, o resultado é cerca de $-7\times 10^5$, sem nenhuma relação com o valor certo.

A causa é o cancelamento da seção 3.3. Os números $\overline{x^2}$ e $\bar x^2$ são da ordem de $10^{20}$ e concordam em quase todos os dígitos. A diferença entre eles, cerca de 9, se perde no arredondamento. O erro relativo cresce como $\varepsilon_{\text{mach}}\,(\bar x/\sigma)^2$: com $\bar x/\sigma \approx 3\times 10^9$, ele passa de $10^{3}$.

### 6.3 A correção

O **algoritmo de Welford** também faz uma passagem só, mas atualiza a média e a soma dos **desvios** a cada novo dado, sem nunca subtrair números grandes:

In [ ]:
def variancia_welford(dados):
    n, media, m2 = 0, 0.0, 0.0
    for x in dados:
        n += 1
        delta = x - media
        media += delta / n
        m2 += delta * (x - media)
    return m2 / (n - 1)

for d in [0.0, 1e6, 1e10]:
    compara(variancia_welford(ruido + d), np.var(ruido + d, ddof=1), rtol=1e-6)

> **Lição.** O código do agente está **matematicamente** correto. O erro é **numérico**, e só aparece com dados realistas. Testes com dados centrados em zero, que são os que um agente costuma escrever, nunca o encontrariam. Teste sempre com dados do tamanho e da escala do problema real, e use invariantes.

## 7. Quando falha

### 7.1 Somar muitos números pequenos

Ao somar números pequenos a um acumulador grande, os bits menos significativos se perdem, como no Patriot. Em precisão simples (`float32`, comum em GPUs e em aprendizado de máquina) isso aparece rápido. A **soma compensada de Kahan** guarda o erro perdido em cada soma e o devolve na próxima.

In [ ]:
def soma_ingenua(valores, tipo):
    s = tipo(0)
    for v in valores:
        s = tipo(s + v)
    return s

def soma_kahan(valores, tipo):
    s, comp = tipo(0), tipo(0)
    for v in valores:
        y = tipo(v - comp)
        t = tipo(s + y)
        comp = tipo((t - s) - y)
        s = t
    return s

valores = np.full(1_000_000, 0.1, dtype=np.float32)
print("valor exato:                 100000")
print("soma ingênua em float32:    ", soma_ingenua(valores, np.float32))
print("soma de Kahan em float32:   ", soma_kahan(valores, np.float32))

### 7.2 Overflow e underflow

O maior `float64` é cerca de $1{,}8\times10^{308}$. Acima disso, o resultado vira `inf`. Um caso frequente é a função **softmax** usada em aprendizado de máquina, $\sigma_i = e^{z_i} / \sum_j e^{z_j}$, que transborda para $z$ grande. Como a expressão não muda se subtrairmos a mesma constante de todos os $z_j$, a versão estável subtrai o máximo:

In [ ]:
z = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(all="ignore"):
    ingenua = np.exp(z) / np.sum(np.exp(z))
estavel = np.exp(z - z.max()) / np.sum(np.exp(z - z.max()))
print("softmax ingênua:", ingenua)
print("softmax estável:", estavel)

### 7.3 Comparar floats com `==`

Como vimos no capítulo 1, `0.1 + 0.2 == 0.3` é falso. Compare sempre com uma tolerância, de preferência relativa: `abs(a - b) <= rtol * max(abs(a), abs(b))`, ou use `math.isclose` e `np.isclose`.

### 7.4 Problema mal condicionado: nenhum algoritmo salva

A função $f(x) = 1 - \cos x$ perto de $x = 0$ sofre cancelamento, mas pode ser reescrita de forma estável como $2\sin^2(x/2)$. Já calcular $\tan x$ perto de $\pi/2$ é **mal condicionado**: $\kappa = |x / (\sin x \cos x)|$ cresce sem limite. Perto de $\pi/2$, mudar $x$ em um único bit, o menor passo possível em ponto flutuante, muda $\tan x$ já na sexta casa: o erro relativo da entrada, cerca de $10^{-16}$, é amplificado $\kappa \approx 10^{10}$ vezes. Nenhuma reescrita resolve, porque o problema é intrinsecamente sensível. O único remédio é saber $x$ com mais precisão.

In [ ]:
x = 1e-8
print(f"1 − cos(x)     = {1 - np.cos(x):.6e}   (cancelamento total)")
print(f"2 sen²(x/2)    = {2 * np.sin(x / 2)**2:.6e}   (estável; o exato é x²/2 = {x**2 / 2:.6e})")
x = np.pi / 2 - 1e-10
x_vizinho = np.nextafter(x, 2.0)            # o próximo float depois de x
print(f"tan(x)           = {np.tan(x):.10e}")
print(f"tan(x + 1 bit)   = {np.tan(x_vizinho):.10e}")
print(f"variação relativa: entrada {(x_vizinho - x) / x:.1e}, saída {(np.tan(x_vizinho) - np.tan(x)) / np.tan(x):.1e}")
print(f"κ = |x / (sen x cos x)| ≈ {abs(x / (np.sin(x) * np.cos(x))):.1e}")

## Resumo

| Fonte de erro | Origem | Como se comporta | Como combater |
| --- | --- | --- | --- |
| Arredondamento | número finito de bits | relativo, ≤ ε/2 ≈ 1,1×10⁻¹⁶ por operação | reescrever a expressão; somas compensadas; mais precisão |
| Cancelamento | subtração de números quase iguais | amplifica o erro relativo | reescrever (conjugado, identidades) |
| Truncamento | processo infinito tornado finito | $O(h^p)$, cai com $h$ | métodos de ordem mais alta |
| Mau condicionamento | o próprio problema | $\kappa \times$ erro da entrada | reformular o problema; não há algoritmo que resolva |
| Instabilidade | o algoritmo | erro muito maior que o inevitável | trocar de algoritmo |

## Exercícios

**Conceituais (sem computador)**

1. Escreva $0{,}2$ e $0{,}25$ em binário. Por que somar 0,25 repetidamente não acumula erro, mas somar 0,1 sim?
2. Calcule o número de condição de $f(x) = \sqrt{x}$, $f(x) = e^x$ e $f(x) = \ln x$. Qual delas é mal condicionada, e perto de que ponto?
3. Mostre que a fórmula alternativa de Bhaskara é algebricamente igual à original.
4. Estime, pela seção 3.5, o passo ótimo e o menor erro possível para a diferença **central**, que tem truncamento $\frac{|f'''|}{6}h^2$. Compare com o gráfico da seção 5.2.

**Verificação (com agente)**

5. Peça a um agente uma função que calcule $e^x$ pela série de Taylor. Teste com $x = -30$. O resultado faz sentido? Explique o cancelamento e peça uma versão estável (dica: $e^{-30} = 1/e^{30}$).
6. Peça a um agente o cálculo do desvio padrão de uma série de dados de um sensor. Escreva o teste do invariante de deslocamento antes de ver o código, e use-o para avaliar a resposta.

**Projeto aberto**

7. **Precisão reduzida em aprendizado de máquina.** Treinamentos modernos usam `float16` e `bfloat16` para economizar memória. Compare a precisão da máquina e a faixa de valores desses formatos com `float32`. Repita a soma da seção 7.1 nesses formatos e explique por que bibliotecas como o PyTorch acumulam somas em precisão maior.

## Leituras

- D. Goldberg, "What every computer scientist should know about floating-point arithmetic", *ACM Computing Surveys* 23(1), 1991.
- N. J. Higham, *Accuracy and Stability of Numerical Algorithms*, 2ª ed., SIAM, 2002, capítulos 1 e 2.
- U.S. General Accounting Office, *Patriot Missile Defense: Software Problem Led to System Failure at Dhahran, Saudi Arabia*, GAO/IMTEC-92-26, 1992.
- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulos 3 e 4.